In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [8]:
train = pd.read_csv("data/train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv("data/test.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

In [10]:
def events_in_window(events, meta):
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    ev = ev[(ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])].copy()
    ev = ev.drop(columns=["window_start_ts", "window_end_ts"])
    return ev

ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)
events_window = pd.concat([ev_train, ev_test], ignore_index=True)

# Девятый блок - Feature Engineering

На основе проведённого EDA сформируем признаки отдельными группами. Каждая функция должна одинаково работать для обучающей и тестовой выборок. Это позволит поэтапно добавлять группы признаков и оценивать их вклад на одной и той же временной валидации

In [1]:
def make_meta_features(meta):
    f = meta[["cookie_id"]].copy()
    f["cookie_age_hours"] = (meta["window_start_ts"] - meta["cookie_created_at"]).dt.total_seconds() / 3600
    f["log_cookie_age_hours"] = np.log1p(f["cookie_age_hours"].clip(lower=0))
    return f

Теперь базовые агрегаты

In [2]:
def make_basic_features(ev, meta):
    g = ev.groupby("cookie_id")

    f = pd.DataFrame(index=g.size().index)
    f["n_events"] = g.size()
    f["event_type_nunique"] = g["event_name"].nunique()
    f["item_nunique"] = g["item_id"].nunique()
    f["category_nunique"] = g["item_category"].nunique()
    f["location_nunique"] = g["item_location"].nunique()
    f["seller_type_nunique"] = g["seller_type"].nunique()

    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)

    return f

Теперь временные признаки

In [3]:
def make_time_features(ev, meta):
    x = ev[["cookie_id", "event_ts"]].copy()
    x = x.sort_values(["cookie_id", "event_ts"])
    x["gap_sec"] = x.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    x["gap_le_5"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 5, np.nan)
    x["gap_le_10"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 10, np.nan)
    x["gap_le_30"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 30, np.nan)

    g = x.groupby("cookie_id")
    f = g.agg(first_event_ts=("event_ts", "min"), last_event_ts=("event_ts", "max"), mean_gap_sec=("gap_sec", "mean"), median_gap_sec=("gap_sec", "median"), std_gap_sec=("gap_sec", "std"), min_gap_sec=("gap_sec", "min"), max_gap_sec=("gap_sec", "max"), share_gap_le_5=("gap_le_5", "mean"), share_gap_le_10=("gap_le_10", "mean"), share_gap_le_30=("gap_le_30", "mean")).reset_index()

    f["active_span_sec"] = (f["last_event_ts"] - f["first_event_ts"]).dt.total_seconds()
    f = f.drop(columns=["first_event_ts", "last_event_ts"])

    event_count = ev.groupby("cookie_id").size().rename("n_events_tmp").reset_index()
    f = f.merge(event_count, on="cookie_id", how="left")
    f["events_per_active_hour"] = np.where(f["active_span_sec"] > 0, (f["n_events_tmp"] - 1) / (f["active_span_sec"] / 3600), 0)
    f = f.drop(columns=["n_events_tmp"])

    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

Теперь очень важная группа , количество каждого типа события

In [4]:
def make_event_count_features(ev, meta):
    counts = ev.groupby(["cookie_id", "event_name"]).size().unstack(fill_value=0)
    counts = counts.add_prefix("count_")

    f = meta[["cookie_id"]].merge(counts.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)

    count_cols = [col for col in f.columns if col.startswith("count_")]
    f["event_count_total"] = f[count_cols].sum(axis=1)

    for col in count_cols:
        f["share_" + col[6:]] = np.where(f["event_count_total"] > 0, f[col] / f["event_count_total"], 0)

    f = f.drop(columns=["event_count_total"])

    return f

Теперь собираем пока только эти четыре семейства

In [5]:
def merge_feature_groups(meta, feature_groups):
    result = meta[["cookie_id"]].copy()

    for features in feature_groups:
        result = result.merge(features, on="cookie_id", how="left")

    return result

Строим трейн и тест

In [11]:
meta_train_features = make_meta_features(train)
meta_test_features = make_meta_features(test)

basic_train_features = make_basic_features(ev_train, train)
basic_test_features = make_basic_features(ev_test, test)

time_train_features = make_time_features(ev_train, train)
time_test_features = make_time_features(ev_test, test)

event_train_features = make_event_count_features(ev_train, train)
event_test_features = make_event_count_features(ev_test, test)

In [13]:
print("Meta:", meta_train_features.shape, meta_test_features.shape)
print("Basic:", basic_train_features.shape, basic_test_features.shape)
print("Time:", time_train_features.shape, time_test_features.shape)
print("Events:", event_train_features.shape, event_test_features.shape)

Meta: (11091, 3) (4909, 3)
Basic: (11091, 7) (4909, 7)
Time: (11091, 11) (4909, 11)
Events: (11091, 19) (4909, 19)


In [12]:
print("NaN meta:", meta_train_features.isna().sum().sum())
print("NaN basic:", basic_train_features.isna().sum().sum())
print("NaN time:", time_train_features.isna().sum().sum())
print("NaN events:", event_train_features.isna().sum().sum())

NaN meta: 0
NaN basic: 0
NaN time: 0
NaN events: 0


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall

In [15]:
def evaluate_feature_set(features, name):
    is_valid = train["window_start_ts"].ge("2026-04-17").values
    cols = [col for col in features.columns if col != "cookie_id"]
    y = train["target"].values

    model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
    model.fit(features.loc[~is_valid, cols], y[~is_valid])

    pred = model.predict_proba(features.loc[is_valid, cols])[:, 1]
    score = precision_at_recall(y[is_valid], pred)
    roc_auc = roc_auc_score(y[is_valid], pred)
    pr_auc = average_precision_score(y[is_valid], pred)

    return {"name": name, "n_features": len(cols), "precision_at_recall_70": score, "roc_auc": roc_auc, "pr_auc": pr_auc}

Сначала воспроизведем бейзлайн

In [16]:
baseline_train_features = basic_train_features[["cookie_id", "n_events", "item_nunique"]].copy()

baseline_result = evaluate_feature_set(baseline_train_features, "baseline")
baseline_result

{'name': 'baseline',
 'n_features': 2,
 'precision_at_recall_70': 0.10247026532479414,
 'roc_auc': 0.6073963567839197,
 'pr_auc': 0.1746089870059772}

# Десятый блок - проверяем разные наборы признаков

Baseline + возраст куки

In [17]:
baseline_meta_features = merge_feature_groups(train, [baseline_train_features, meta_train_features])

baseline_meta_result = evaluate_feature_set(baseline_meta_features, "baseline + meta")
baseline_meta_result

{'name': 'baseline + meta',
 'n_features': 4,
 'precision_at_recall_70': 0.13023255813953488,
 'roc_auc': 0.668606923506421,
 'pr_auc': 0.1910817488399391}

Baseline + временное поведение

In [18]:
baseline_time_features = merge_feature_groups(train, [baseline_train_features, time_train_features])

baseline_time_result = evaluate_feature_set(baseline_time_features, "baseline + time")
baseline_time_result

{'name': 'baseline + time',
 'n_features': 12,
 'precision_at_recall_70': 0.338368580060423,
 'roc_auc': 0.844550879396985,
 'pr_auc': 0.5640448646414734}

Baseline + состав событий

In [19]:
baseline_event_features = merge_feature_groups(train, [baseline_train_features, event_train_features])

baseline_event_result = evaluate_feature_set(baseline_event_features, "baseline + events")
baseline_event_result

{'name': 'baseline + events',
 'n_features': 20,
 'precision_at_recall_70': 0.1527027027027027,
 'roc_auc': 0.7447236180904523,
 'pr_auc': 0.3645483917060856}

И все вместе

In [20]:
first_full_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features])

first_full_result = evaluate_feature_set(first_full_features, "basic + meta + time + events")
first_full_result

{'name': 'basic + meta + time + events',
 'n_features': 36,
 'precision_at_recall_70': 0.38620689655172413,
 'roc_auc': 0.8685755164712452,
 'pr_auc': 0.637099987490662}

Продолажем

In [21]:
def make_item_features(ev, meta):
    item_view = ev[ev["event_name"] == "item_view"].copy()

    f = item_view.groupby("cookie_id").agg(item_view_count=("item_id", "size"), item_view_nunique=("item_id", "nunique")).reset_index()
    f["item_view_repeat_count"] = f["item_view_count"] - f["item_view_nunique"]
    f["item_view_unique_ratio"] = np.where(f["item_view_count"] > 0, f["item_view_nunique"] / f["item_view_count"], 0)

    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

In [22]:
item_train_features = make_item_features(ev_train, train)
item_test_features = make_item_features(ev_test, test)

print(item_train_features.shape, item_test_features.shape)
print("NaN train:", item_train_features.isna().sum().sum())
print("NaN test:", item_test_features.isna().sum().sum())

(11091, 5) (4909, 5)
NaN train: 0
NaN test: 0


In [23]:
def make_search_features(ev, meta):
    search = ev[ev["event_name"] == "search_results_view"].copy()

    f = search.groupby("cookie_id").agg(search_query_nunique=("search_query", "nunique"), search_page_mean=("search_page", "mean"), search_page_median=("search_page", "median"), search_page_max=("search_page", "max"), search_page_nunique=("search_page", "nunique")).reset_index()

    search_count = search.groupby("cookie_id").size().rename("search_event_count").reset_index()
    f = f.merge(search_count, on="cookie_id", how="outer")
    f["searches_per_query"] = np.where(f["search_query_nunique"] > 0, f["search_event_count"] / f["search_query_nunique"], 0)

    search["page_gt_1"] = search["search_page"] > 1
    search["page_gt_3"] = search["search_page"] > 3
    search["page_gt_5"] = search["search_page"] > 5

    depth = search.groupby("cookie_id").agg(share_page_gt_1=("page_gt_1", "mean"), share_page_gt_3=("page_gt_3", "mean"), share_page_gt_5=("page_gt_5", "mean")).reset_index()
    f = f.merge(depth, on="cookie_id", how="outer")

    query_page = search.dropna(subset=["search_query", "search_page"]).drop_duplicates(["cookie_id", "search_query", "search_page"])
    query_page_nunique = query_page.groupby("cookie_id").size().rename("query_page_nunique").reset_index()
    f = f.merge(query_page_nunique, on="cookie_id", how="outer")

    pages_per_query = query_page.groupby(["cookie_id", "search_query"]).size().rename("pages_for_query").reset_index()
    pages_summary = pages_per_query.groupby("cookie_id")["pages_for_query"].agg(["mean", "max"]).reset_index()
    pages_summary.columns = ["cookie_id", "pages_per_query_mean", "pages_per_query_max"]
    f = f.merge(pages_summary, on="cookie_id", how="outer")

    f["unique_query_page_ratio"] = np.where(f["search_event_count"] > 0, f["query_page_nunique"] / f["search_event_count"], 0)

    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

In [24]:
search_train_features = make_search_features(ev_train, train)
search_test_features = make_search_features(ev_test, test)

print(search_train_features.shape, search_test_features.shape)
print("NaN train:", search_train_features.isna().sum().sum())
print("NaN test:", search_test_features.isna().sum().sum())

(11091, 15) (4909, 15)
NaN train: 0
NaN test: 0


Далее продолжаем
baseline + items

baseline + search 

все + items, search

In [25]:
baseline_item_features = merge_feature_groups(train, [baseline_train_features, item_train_features])

baseline_item_result = evaluate_feature_set(baseline_item_features, "baseline + item")
baseline_item_result

{'name': 'baseline + item',
 'n_features': 6,
 'precision_at_recall_70': 0.11460446247464504,
 'roc_auc': 0.6528091848129537,
 'pr_auc': 0.22876348973920824}

In [26]:
baseline_search_features = merge_feature_groups(train, [baseline_train_features, search_train_features])

baseline_search_result = evaluate_feature_set(baseline_search_features, "baseline + search")
baseline_search_result

{'name': 'baseline + search',
 'n_features': 16,
 'precision_at_recall_70': 0.1324736225087925,
 'roc_auc': 0.7322026800670017,
 'pr_auc': 0.3555960524931956}

In [27]:
full_item_search_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, item_train_features, search_train_features])

full_item_search_result = evaluate_feature_set(full_item_search_features, "full + item + search")
full_item_search_result

{'name': 'full + item + search',
 'n_features': 54,
 'precision_at_recall_70': 0.4409448818897638,
 'roc_auc': 0.8596105527638191,
 'pr_auc': 0.6242951931140025}

Увидел интересные результаты. Сами по себе признаки почти бесполезны, а вот в сочетании с событиями - очень даже хорошо.
Теперь время сделать временную кросс-валидацию.

In [28]:
temporal_folds = [
    ("2026-04-11", "2026-04-13"),
    ("2026-04-14", "2026-04-16"),
    ("2026-04-17", "2026-04-19")
]

In [29]:
def evaluate_temporal_folds(features, name):
    cols = [col for col in features.columns if col != "cookie_id"]
    y = train["target"].values
    dates = train["window_start_ts"]
    fold_results = []

    for fold_id, (valid_start, valid_end) in enumerate(temporal_folds, start=1):
        train_mask = dates < valid_start
        valid_mask = dates.between(valid_start, valid_end)

        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
        model.fit(features.loc[train_mask, cols], y[train_mask])

        pred = model.predict_proba(features.loc[valid_mask, cols])[:, 1]

        score = precision_at_recall(y[valid_mask], pred)
        roc_auc = roc_auc_score(y[valid_mask], pred)
        pr_auc = average_precision_score(y[valid_mask], pred)

        fold_results.append({"name": name, "fold": fold_id, "valid_start": valid_start, "valid_end": valid_end, "n_train": train_mask.sum(), "n_valid": valid_mask.sum(), "n_positive": y[valid_mask].sum(), "precision_at_recall_70": score, "roc_auc": roc_auc, "pr_auc": pr_auc})

    return pd.DataFrame(fold_results)

In [30]:
baseline_cv = evaluate_temporal_folds(baseline_train_features, "baseline")
baseline_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,baseline,1,2026-04-11,2026-04-13,4217,2556,199,0.093144,0.639692,0.236794
1,baseline,2,2026-04-14,2026-04-16,6773,2367,195,0.104677,0.637447,0.235197
2,baseline,3,2026-04-17,2026-04-19,9140,1951,160,0.102470,0.607396,0.174609


In [31]:
first_full_cv = evaluate_temporal_folds(first_full_features, "basic + meta + time + events")
first_full_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,basic + meta + time + events,1,2026-04-11,2026-04-13,4217,2556,199,0.316027,0.839843,0.624602
1,basic + meta + time + events,2,2026-04-14,2026-04-16,6773,2367,195,0.348371,0.861610,0.640661
2,basic + meta + time + events,3,2026-04-17,2026-04-19,9140,1951,160,0.386207,0.868576,0.637100


In [32]:
full_item_search_cv = evaluate_temporal_folds(full_item_search_features, "full + item + search")
full_item_search_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,full + item + search,1,2026-04-11,2026-04-13,4217,2556,199,0.280561,0.848018,0.619413
1,full + item + search,2,2026-04-14,2026-04-16,6773,2367,195,0.290254,0.857317,0.627809
2,full + item + search,3,2026-04-17,2026-04-19,9140,1951,160,0.440945,0.859611,0.624295


In [34]:
cv_pivot = cv_results.pivot(index="fold", columns="name", values="precision_at_recall_70")
cv_pivot

name,baseline,basic + meta + time + events,full + item + search
fold,,,
1,0.093144,0.316027,0.280561
2,0.104677,0.348371,0.290254
3,0.102470,0.386207,0.440945


item/search сильно помогают только на самом последнем временном отрезке, но ухудшают первые два

Дальше будем добавлять признаки к к устойчивому набору

In [35]:
def make_tech_features(ev, meta):
    x = ev[["cookie_id", "platform", "user_agent"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["ua_lower"] = x["user_agent"].str.lower()

    x["is_headless"] = x["ua_lower"].str.contains("headlesschrome")
    x["is_firefox"] = x["ua_lower"].str.contains("firefox")
    x["is_yandex"] = x["ua_lower"].str.contains("yabrowser")
    x["is_avito_app"] = x["ua_lower"].str.startswith("avito/")
    x["is_chrome"] = x["ua_lower"].str.contains("chrome") & ~x["is_headless"] & ~x["is_yandex"]

    x["is_windows"] = x["ua_lower"].str.contains("windows")
    x["is_linux"] = x["ua_lower"].str.contains("linux") & ~x["ua_lower"].str.contains("android")
    x["is_macos"] = x["ua_lower"].str.contains("macintosh")
    x["is_android_ua"] = x["ua_lower"].str.contains("android")

    x["ua_length"] = x["user_agent"].str.len()

    f = x.groupby("cookie_id").agg(platform_nunique=("platform_norm", "nunique"), ua_nunique=("user_agent", "nunique"), ua_length_mean=("ua_length", "mean"), ua_length_min=("ua_length", "min"), ua_length_max=("ua_length", "max"), is_headless=("is_headless", "max"), is_firefox=("is_firefox", "max"), is_yandex=("is_yandex", "max"), is_avito_app=("is_avito_app", "max"), is_chrome=("is_chrome", "max"), is_windows=("is_windows", "max"), is_linux=("is_linux", "max"), is_macos=("is_macos", "max"), is_android_ua=("is_android_ua", "max")).reset_index()

    platform_counts = x.groupby(["cookie_id", "platform_norm"]).size().unstack(fill_value=0)
    platform_counts = platform_counts.div(platform_counts.sum(axis=1), axis=0)
    platform_counts = platform_counts.add_prefix("platform_share_").reset_index()

    f = f.merge(platform_counts, on="cookie_id", how="left")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

In [36]:
tech_train_features = make_tech_features(ev_train, train)
tech_test_features = make_tech_features(ev_test, test)

print(tech_train_features.shape, tech_test_features.shape)
print("NaN train:", tech_train_features.isna().sum().sum())
print("NaN test:", tech_test_features.isna().sum().sum())

(11091, 20) (4909, 20)
NaN train: 0
NaN test: 0


In [37]:
baseline_tech_features = merge_feature_groups(train, [baseline_train_features, tech_train_features])

baseline_tech_result = evaluate_feature_set(baseline_tech_features, "baseline + tech")
baseline_tech_result

{'name': 'baseline + tech',
 'n_features': 21,
 'precision_at_recall_70': 0.11177644710578842,
 'roc_auc': 0.66812360413177,
 'pr_auc': 0.22156460552777157}

In [38]:
core_tech_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features])

core_tech_result = evaluate_feature_set(core_tech_features, "core + tech")
core_tech_result

{'name': 'core + tech',
 'n_features': 55,
 'precision_at_recall_70': 0.45161290322580644,
 'roc_auc': 0.8669161781127861,
 'pr_auc': 0.6467667972663448}

In [39]:
core_tech_cv = evaluate_temporal_folds(core_tech_features, "core + tech")
core_tech_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,core + tech,1,2026-04-11,2026-04-13,4217,2556,199,0.301075,0.845436,0.629709
1,core + tech,2,2026-04-14,2026-04-16,6773,2367,195,0.337438,0.862240,0.642581
2,core + tech,3,2026-04-17,2026-04-19,9140,1951,160,0.451613,0.866916,0.646767


core + tech поднимает средний P@R70 примерно с 0.350 до 0.363, но стандартное отклонение растёт примерно с 0.035 до 0.079. То есть технические признаки полезны, но их эффект нестабилен

In [40]:
def make_pointer_features(ev, meta):
    x = ev[["cookie_id", "event_ts", "platform", "pointer_x", "pointer_y"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["has_pointer"] = x["pointer_x"].notna() & x["pointer_y"].notna()

    basic = x.groupby("cookie_id").agg(pointer_event_count=("has_pointer", "sum"), pointer_event_share=("has_pointer", "mean")).reset_index()
    basic["has_any_pointer"] = basic["pointer_event_count"] > 0

    valid = x[x["has_pointer"]].copy()

    coord = valid.groupby("cookie_id").agg(pointer_x_mean=("pointer_x", "mean"), pointer_x_std=("pointer_x", "std"), pointer_x_min=("pointer_x", "min"), pointer_x_max=("pointer_x", "max"), pointer_y_mean=("pointer_y", "mean"), pointer_y_std=("pointer_y", "std"), pointer_y_min=("pointer_y", "min"), pointer_y_max=("pointer_y", "max")).reset_index()

    coord["pointer_x_range"] = coord["pointer_x_max"] - coord["pointer_x_min"]
    coord["pointer_y_range"] = coord["pointer_y_max"] - coord["pointer_y_min"]

    valid["_row_order"] = np.arange(len(valid))
    valid = valid.sort_values(["cookie_id", "event_ts", "_row_order"])

    valid["prev_x"] = valid.groupby("cookie_id")["pointer_x"].shift()
    valid["prev_y"] = valid.groupby("cookie_id")["pointer_y"].shift()
    valid["dx"] = valid["pointer_x"] - valid["prev_x"]
    valid["dy"] = valid["pointer_y"] - valid["prev_y"]
    valid["move_distance"] = np.sqrt(valid["dx"] ** 2 + valid["dy"] ** 2)

    moves = valid.groupby("cookie_id").agg(move_distance_mean=("move_distance", "mean"), move_distance_median=("move_distance", "median"), move_distance_std=("move_distance", "std"), move_distance_max=("move_distance", "max")).reset_index()

    pairs = valid.drop_duplicates(["cookie_id", "pointer_x", "pointer_y"]).groupby("cookie_id").size().rename("pointer_pair_nunique").reset_index()
    counts = valid.groupby("cookie_id").size().rename("pointer_count").reset_index()

    repeat = counts.merge(pairs, on="cookie_id", how="left")
    repeat["pointer_unique_ratio"] = np.where(repeat["pointer_count"] > 0, repeat["pointer_pair_nunique"] / repeat["pointer_count"], 0)

    platform_pointer = x.groupby(["cookie_id", "platform_norm"])["has_pointer"].mean().unstack(fill_value=0)
    platform_pointer = platform_pointer.add_prefix("pointer_share_").reset_index()

    f = basic.merge(coord, on="cookie_id", how="left")
    f = f.merge(moves, on="cookie_id", how="left")
    f = f.merge(repeat[["cookie_id", "pointer_unique_ratio"]], on="cookie_id", how="left")
    f = f.merge(platform_pointer, on="cookie_id", how="left")

    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

In [41]:
pointer_train_features = make_pointer_features(ev_train, train)
pointer_test_features = make_pointer_features(ev_test, test)

print(pointer_train_features.shape, pointer_test_features.shape)
print("NaN train:", pointer_train_features.isna().sum().sum())
print("NaN test:", pointer_test_features.isna().sum().sum())

(11091, 24) (4909, 24)
NaN train: 0
NaN test: 0


Сначала по указателю отдельно

In [42]:
baseline_pointer_features = merge_feature_groups(train, [baseline_train_features, pointer_train_features])

baseline_pointer_result = evaluate_feature_set(baseline_pointer_features, "baseline + pointer")
baseline_pointer_result

{'name': 'baseline + pointer',
 'n_features': 25,
 'precision_at_recall_70': 0.1755485893416928,
 'roc_auc': 0.7927781267448353,
 'pr_auc': 0.38995236770102093}

Потом указатель поверх устойчивого набора

In [43]:
core_pointer_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, pointer_train_features])

core_pointer_result = evaluate_feature_set(core_pointer_features, "core + pointer")
core_pointer_result

{'name': 'core + pointer',
 'n_features': 59,
 'precision_at_recall_70': 0.70625,
 'roc_auc': 0.9168341708542713,
 'pr_auc': 0.7340144495433972}

In [44]:
core_pointer_cv = evaluate_temporal_folds(core_pointer_features, "core + pointer")
core_pointer_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,core + pointer,1,2026-04-11,2026-04-13,4217,2556,199,0.598291,0.893980,0.702368
1,core + pointer,2,2026-04-14,2026-04-16,6773,2367,195,0.600877,0.910018,0.721303
2,core + pointer,3,2026-04-17,2026-04-19,9140,1951,160,0.706250,0.916834,0.734014


In [45]:
core_tech_pointer_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features])

core_tech_pointer_result = evaluate_feature_set(core_tech_pointer_features, "core + tech + pointer")
core_tech_pointer_result

{'name': 'core + tech + pointer',
 'n_features': 78,
 'precision_at_recall_70': 0.7151898734177216,
 'roc_auc': 0.9164817141261865,
 'pr_auc': 0.7417469038437443}

In [46]:
core_tech_pointer_cv = evaluate_temporal_folds(core_tech_pointer_features, "core + tech + pointer")
core_tech_pointer_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,core + tech + pointer,1,2026-04-11,2026-04-13,4217,2556,199,0.642202,0.898569,0.714003
1,core + tech + pointer,2,2026-04-14,2026-04-16,6773,2367,195,0.611607,0.910410,0.728584
2,core + tech + pointer,3,2026-04-17,2026-04-19,9140,1951,160,0.715190,0.916482,0.741747


Указатель оказался не просто полезным, а крупнейшим источником прироста после временных признаков

Следующим логично проверить sequence-признаки

In [47]:
def make_sequence_features(ev, meta):
    x = ev[["cookie_id", "event_ts", "event_name"]].copy()
    x["_row_order"] = np.arange(len(x))
    x = x.sort_values(["cookie_id", "event_ts", "_row_order"])

    x["next_event"] = x.groupby("cookie_id")["event_name"].shift(-1)
    x["same_event_transition"] = x["event_name"] == x["next_event"]

    transitions = x.dropna(subset=["next_event"]).copy()
    transition_summary = transitions.groupby("cookie_id").agg(transition_count=("next_event", "size"), same_event_transition_share=("same_event_transition", "mean")).reset_index()

    transitions["transition"] = transitions["event_name"] + " -> " + transitions["next_event"]
    transition_nunique = transitions.groupby("cookie_id")["transition"].nunique().rename("transition_nunique").reset_index()
    transition_summary = transition_summary.merge(transition_nunique, on="cookie_id", how="left")
    transition_summary["transition_unique_ratio"] = np.where(transition_summary["transition_count"] > 0, transition_summary["transition_nunique"] / transition_summary["transition_count"], 0)

    x["new_run"] = x["event_name"] != x.groupby("cookie_id")["event_name"].shift()
    x["run_id"] = x.groupby("cookie_id")["new_run"].cumsum()

    run_lengths = x.groupby(["cookie_id", "run_id", "event_name"]).size().rename("run_length").reset_index()
    run_summary = run_lengths.groupby("cookie_id")["run_length"].agg(["mean", "max"]).reset_index()
    run_summary.columns = ["cookie_id", "run_length_mean", "run_length_max"]

    run_event_max = run_lengths.pivot_table(index="cookie_id", columns="event_name", values="run_length", aggfunc="max", fill_value=0)
    run_event_max = run_event_max.add_prefix("max_run_").reset_index()

    event_distribution = x.groupby(["cookie_id", "event_name"]).size().rename("event_count").reset_index()
    event_distribution["total_events"] = event_distribution.groupby("cookie_id")["event_count"].transform("sum")
    event_distribution["event_prob"] = event_distribution["event_count"] / event_distribution["total_events"]
    event_distribution["entropy_component"] = -event_distribution["event_prob"] * np.log(event_distribution["event_prob"])

    entropy = event_distribution.groupby("cookie_id")["entropy_component"].sum().rename("event_entropy").reset_index()
    event_nunique = event_distribution.groupby("cookie_id").size().rename("sequence_event_type_nunique").reset_index()
    entropy = entropy.merge(event_nunique, on="cookie_id", how="left")
    entropy["normalized_event_entropy"] = np.where(entropy["sequence_event_type_nunique"] > 1, entropy["event_entropy"] / np.log(entropy["sequence_event_type_nunique"]), 0)

    f = transition_summary.merge(run_summary, on="cookie_id", how="outer")
    f = f.merge(run_event_max, on="cookie_id", how="outer")
    f = f.merge(entropy, on="cookie_id", how="outer")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)

    return f

In [48]:
sequence_train_features = make_sequence_features(ev_train, train)
sequence_test_features = make_sequence_features(ev_test, test)

print(sequence_train_features.shape, sequence_test_features.shape)
print("NaN train:", sequence_train_features.isna().sum().sum())
print("NaN test:", sequence_test_features.isna().sum().sum())

(11091, 19) (4909, 19)
NaN train: 0
NaN test: 0


In [49]:
print("Совпадают колонки:", sequence_train_features.columns.tolist() == sequence_test_features.columns.tolist())
print(sequence_train_features.columns.tolist())

Совпадают колонки: True
['cookie_id', 'transition_count', 'same_event_transition_share', 'transition_nunique', 'transition_unique_ratio', 'run_length_mean', 'run_length_max', 'max_run_contact_chat_open', 'max_run_contact_message_sent', 'max_run_contact_phone_show', 'max_run_favorite_add', 'max_run_item_view', 'max_run_login', 'max_run_photo_swipe', 'max_run_search_results_view', 'max_run_seller_page_view', 'event_entropy', 'sequence_event_type_nunique', 'normalized_event_entropy']


In [50]:
baseline_sequence_features = merge_feature_groups(train, [baseline_train_features, sequence_train_features])

baseline_sequence_result = evaluate_feature_set(baseline_sequence_features, "baseline + sequence")
baseline_sequence_result

{'name': 'baseline + sequence',
 'n_features': 20,
 'precision_at_recall_70': 0.14947089947089948,
 'roc_auc': 0.7458385678391961,
 'pr_auc': 0.3147578217447716}

In [51]:
core_tech_pointer_sequence_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features, sequence_train_features])

core_tech_pointer_sequence_result = evaluate_feature_set(core_tech_pointer_sequence_features, "core + tech + pointer + sequence")
core_tech_pointer_sequence_result

{'name': 'core + tech + pointer + sequence',
 'n_features': 96,
 'precision_at_recall_70': 0.7272727272727273,
 'roc_auc': 0.9146531267448352,
 'pr_auc': 0.7320764507448102}

In [52]:
core_tech_pointer_sequence_cv = evaluate_temporal_folds(core_tech_pointer_sequence_features, "core + tech + pointer + sequence")
core_tech_pointer_sequence_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,core + tech + pointer + sequence,1,2026-04-11,2026-04-13,4217,2556,199,0.606061,0.892282,0.700330
1,core + tech + pointer + sequence,2,2026-04-14,2026-04-16,6773,2367,195,0.602620,0.909653,0.717917
2,core + tech + pointer + sequence,3,2026-04-17,2026-04-19,9140,1951,160,0.727273,0.914653,0.732076


Теперь проверим, что происходит, если к уже сильной модели добавить item + search

In [53]:
core_tech_pointer_item_search_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features, item_train_features, search_train_features])

core_tech_pointer_item_search_result = evaluate_feature_set(core_tech_pointer_item_search_features, "core + tech + pointer + item + search")
core_tech_pointer_item_search_result

{'name': 'core + tech + pointer + item + search',
 'n_features': 96,
 'precision_at_recall_70': 0.7320261437908496,
 'roc_auc': 0.9203063930764935,
 'pr_auc': 0.7438284264032017}

In [54]:
core_tech_pointer_item_search_cv = evaluate_temporal_folds(core_tech_pointer_item_search_features, "core + tech + pointer + item + search")
core_tech_pointer_item_search_cv

,name,fold,valid_start,valid_end,n_train,n_valid,n_positive,precision_at_recall_70,roc_auc,pr_auc
0,core + tech + pointer + item + search,1,2026-04-11,2026-04-13,4217,2556,199,0.648148,0.901134,0.710264
1,core + tech + pointer + item + search,2,2026-04-14,2026-04-16,6773,2367,195,0.594017,0.906539,0.725790
2,core + tech + pointer + item + search,3,2026-04-17,2026-04-19,9140,1951,160,0.732026,0.920306,0.743828


И проверим все признаки

In [55]:
all_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features, item_train_features, search_train_features, sequence_train_features])

all_result = evaluate_feature_set(all_features, "all features")
all_result

{'name': 'all features',
 'n_features': 114,
 'precision_at_recall_70': 0.691358024691358,
 'roc_auc': 0.9122539782244556,
 'pr_auc': 0.7308323874823949}